# Bölüm klipleri: ücretsiz ekran kartında hareketli video

Bu not defteri başlangıç karelerini **LTX-Video** modeliyle hareketli kliplere çevirir.
Kareler ya `npm run kaggle:paket -- --kareler` zip'inden ya da `kare-uret` not defterinin çıktısından gelir.

**Ayarlar (sağdaki panel → Session options):**
- Accelerator: **GPU T4 x2** (ya da GPU P100)
- Internet: **On** (telefon doğrulaması gerekir)
- Sağ üstte **Add Input** → kare zip'ini (dataset) ya da `kare-uret` not defterinin çıktısını ekleyin.

Sonra üstteki **Run All** düğmesine basın. Bitince sağdaki **Output** bölümünden
`klipler.zip` dosyasını indirin.

Bir oturum yarıda kalırsa tekrar **Run All** deyin: biten klipler `/kaggle/working/clips`
içinde durduğu sürece tekrar üretilmez.

In [ ]:
# 1) Kurulum (yaklaşık 2-3 dakika)
!pip install -q -U "diffusers>=0.34" "transformers>=4.48" accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
# 2) Ayarlar
import glob, json, os, time

# Kare zip'i Kaggle'da /kaggle/input/<dataset-adı>/ altına açılır; plan.json'u kendimiz buluyoruz.
plans = glob.glob("/kaggle/input/**/plan.json", recursive=True)
assert plans, "plan.json bulunamadı. Sağ üstte 'Add Input' ile kare zip'ini (dataset) ekleyin."
PLAN_PATH = plans[0]
plan = json.load(open(PLAN_PATH))
# Karelerin olduğu klasör (kare-uret çıktısı ayrı bir girdi olabilir)
firsts = glob.glob(f"/kaggle/input/**/keyframes/{plan['clips'][0]['key']}.png", recursive=True)
assert firsts, "Kareler bulunamadı. Önce kare-uret not defterini çalıştırıp çıktısını girdi olarak ekleyin."
DATA_DIR = os.path.dirname(os.path.dirname(firsts[0]))

OUT_DIR = "/kaggle/working/clips"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_ID = "Lightricks/LTX-Video-0.9.7-distilled"
WIDTH, HEIGHT = 832, 480        # 32'nin katı olmalı; 16:9'a yakın
FPS = 24
STEPS = 8                       # distilled model az adımla çalışır
SEED = 23

# Yalnızca bazı sahneleri üretmek için: ONLY = ["c03", "c04"]
ONLY = None

print(f"{plan['series']} · Bölüm {plan['episode']} · {len(plan['clips'])} sahne")
print("Kareler:", DATA_DIR)

In [ ]:
# 3) Modeli yükle (ilk seferde ~10 GB indirir)
import torch
from diffusers import LTXConditionPipeline
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from diffusers.utils import export_to_video, load_image

gpu = torch.cuda.get_device_name(0)
# T4 ve P100 bfloat16'yı donanımda desteklemez; bu kartlarda float16 kullanıyoruz.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("Ekran kartı:", gpu, "·", DTYPE)

pipe = LTXConditionPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()   # 16 GB'lık kartta metin kodlayıcı + model birlikte sığmaz
pipe.vae.enable_tiling()

In [ ]:
# 4) Klipleri üret
NEGATIVE = "worst quality, inconsistent motion, blurry, jittery, distorted, deformed face, extra limbs, watermark, text"

def frames_for(seconds):
    # LTX kare sayısı 8k+1 olmalı
    n = int(seconds * FPS)
    return (n // 8) * 8 + 1

done, failed = [], []
for c in plan["clips"]:
    key = c["key"]
    if ONLY and key not in ONLY:
        continue
    out = os.path.join(OUT_DIR, f"{key}.mp4")
    if os.path.exists(out):
        print(f"✔ {key} zaten var, atlandı")
        continue
    t0 = time.time()
    try:
        image = load_image(os.path.join(DATA_DIR, "keyframes", f"{key}.png")).resize((WIDTH, HEIGHT))
        # 10 saniyelik sahnelerde uzun video belleği aşar; 5 sn üretip birleştirmede yavaşlatıyoruz.
        seconds = min(c["duration"], 5)
        prompt = f"{c['motion']}. {plan['style']}"
        video = pipe(
            conditions=[LTXVideoCondition(image=image, frame_index=0)],
            prompt=prompt,
            negative_prompt=NEGATIVE,
            width=WIDTH,
            height=HEIGHT,
            num_frames=frames_for(seconds),
            num_inference_steps=STEPS,
            guidance_scale=1.0,
            decode_timestep=0.05,
            decode_noise_scale=0.025,
            image_cond_noise_scale=0.0,
            generator=torch.Generator().manual_seed(SEED),
            output_type="pil",
        ).frames[0]
        export_to_video(video, out, fps=FPS)
        done.append(key)
        print(f"✔ {key}  {time.time() - t0:.0f} sn")
    except Exception as e:
        failed.append(key)
        print(f"✖ {key}: {e}")
    torch.cuda.empty_cache()

print(f"\nBiten: {len(done)} · Hatalı: {failed or 'yok'}")

In [ ]:
# 5) 10 saniyelik sahneleri yavaşlatıp tam süreye getir, sonra hepsini zip'le
import subprocess, shutil

for c in plan["clips"]:
    src = os.path.join(OUT_DIR, f"{c['key']}.mp4")
    if c["duration"] <= 5 or not os.path.exists(src) or src.endswith("-slow.mp4"):
        continue
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", src],
        capture_output=True, text=True,
    ).stdout.strip()
    if probe and float(probe) >= c["duration"] - 0.5:
        continue  # zaten tam süre
    factor = c["duration"] / float(probe)
    tmp = src.replace(".mp4", "-tmp.mp4")
    # Kare enterpolasyonuyla yumuşak yavaşlatma
    subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", src,
        "-vf", f"setpts={factor}*PTS,minterpolate=fps={FPS}:mi_mode=mci",
        "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", tmp,
    ], check=True)
    os.replace(tmp, src)
    print(f"↔ {c['key']}: {float(probe):.1f} sn → {c['duration']} sn")

zip_base = f"/kaggle/working/klipler-{plan['series']}-e{plan['episode']}"
shutil.make_archive(zip_base, "zip", OUT_DIR)
print("\n🎬 İndirilecek dosya:", zip_base + ".zip")
print("Sağdaki Output bölümünden indirin ve bilgisayarınızda 'npm run kaggle:al' ile yerleştirin.")